# LightGBM Waste Forecasting

In [23]:
!pip install lightgbm optuna


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [24]:
import os
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

RANDOM_SEED   = 42
np.random.seed(RANDOM_SEED)
optuna.logging.set_verbosity(optuna.logging.WARNING)
os.makedirs('models/lightgbm_optimized', exist_ok=True)

MEALS  = ['breakfast', 'lunch', 'dinner']
TARGET = 'waste_kg'
SECTIONS = ['a', 'b', 'c', 'd']

## Load and Inspect Dataset

In [25]:
def load_and_filter(path: str, meals: list) -> pd.DataFrame:
    """Load raw CSV and retain only the three operational meal types."""
    df = pd.read_csv(path)
    print(f'Raw dataset shape: {df.shape}')
    print('Meal values:', df['meal'].unique())
    df = df[df['meal'].isin(meals)].copy()
    print(f'After filtering to {meals}: {df.shape}')
    return df

df_raw = load_and_filter('data/waste_features_xgb.csv', MEALS)
df_raw.head()


Raw dataset shape: (2506, 33)
Meal values: <ArrowStringArray>
['lunch', 'breakfast', 'closed', 'dinner']
Length: 4, dtype: str
After filtering to ['breakfast', 'lunch', 'dinner']: (1813, 33)


,meal,waste_kg,waste_organic_kg,waste_recyclable_kg,waste_landfill_kg,foot_traffic,is_holiday,has_special_event,year,month,...,lag_28,rolling_mean_7,rolling_mean_14,rolling_std_7,rolling_max_7,section_encoded,section_a,section_b,section_c,section_d
0,lunch,5.17,2.74,1.75,0.68,102.00,0,1,2025,1,...,0.18,2.262857,2.102857,2.249257,5.58,0,True,False,False,False
1,breakfast,4.26,1.74,1.27,1.24,46.25,0,1,2025,1,...,3.18,2.854286,2.396429,2.133197,5.58,0,True,False,False,False
3,dinner,2.72,1.72,0.75,0.25,75.00,0,1,2025,1,...,0.06,2.147143,2.390000,1.978819,5.17,0,True,False,False,False
4,lunch,6.07,3.12,2.06,0.89,138.00,0,1,2025,1,...,1.50,2.842857,2.810000,2.401317,6.07,0,True,False,False,False
5,breakfast,2.96,1.21,0.89,0.85,46.00,0,1,2025,1,...,1.82,3.244286,2.786429,2.090940,6.07,0,True,False,False,False


## Preprocess Data

In [26]:
def identify_section(row) -> str|None:
    """Return the section label (a–d) for a given row."""
    for s in SECTIONS:
        if row.get(f'section_{s}', False):
            return s
    return None


def compute_lag_rolling_features(df: pd.DataFrame, target: str,
                                  group_key: list) -> pd.DataFrame:
    for lag in [1, 7, 14, 28]:
        df[f'lag_{lag}'] = df.groupby(group_key)[target].shift(lag)

    df['rolling_mean_7']  = df.groupby(group_key)[target].transform(
        lambda x: x.shift(1).rolling(7,  min_periods=1).mean())
    df['rolling_mean_14'] = df.groupby(group_key)[target].transform(
        lambda x: x.shift(1).rolling(14, min_periods=1).mean())
    df['rolling_std_7']   = df.groupby(group_key)[target].transform(
        lambda x: x.shift(1).rolling(7,  min_periods=1).std().fillna(0))
    df['rolling_max_7']   = df.groupby(group_key)[target].transform(
        lambda x: x.shift(1).rolling(7,  min_periods=1).max())
    return df


def split_by_section(df: pd.DataFrame, sections: list) -> dict:
    """Return {section_label: DataFrame} with indicator/encoded columns dropped."""
    drop_cols = [f'section_{s}' for s in sections] + ['section_encoded', 'section']
    result = {}
    for sec in sections:
        sdf = df[df['section'] == sec].copy()
        sdf = sdf.drop(columns=drop_cols, errors='ignore')
        result[sec] = sdf
        print(f'Section {sec.upper()}: {len(sdf)} rows, '
              f'{sdf["meal_type"].value_counts().to_dict()}')
    return result


df_raw['date']     = pd.to_datetime(df_raw[['year', 'month', 'day']])
df_raw['section']  = df_raw.apply(identify_section, axis=1)
df_raw             = df_raw.rename(columns={'meal': 'meal_type'})
df_raw             = df_raw.sort_values(['section', 'date', 'meal_type']).reset_index(drop=True)

GROUP_KEY = ['section', 'meal_type']
df_raw    = compute_lag_rolling_features(df_raw, TARGET, GROUP_KEY)
df_raw    = df_raw.dropna(subset=[f'lag_{l}' for l in [1, 7, 14, 28]]).reset_index(drop=True)
print(f'After preprocessing: {df_raw.shape}')

section_data = split_by_section(df_raw, SECTIONS)


After preprocessing: (1477, 35)
Section A: 371 rows, {'dinner': 127, 'breakfast': 125, 'lunch': 119}
Section B: 372 rows, {'dinner': 130, 'breakfast': 125, 'lunch': 117}
Section C: 372 rows, {'dinner': 131, 'breakfast': 129, 'lunch': 112}
Section D: 362 rows, {'dinner': 131, 'breakfast': 124, 'lunch': 107}


## Define Features and Target

`meal_type` is kept as a LightGBM categorical feature (encoded via `category` dtype).
`date` is dropped before training.


In [27]:
def get_X_y(data: pd.DataFrame, target: str = TARGET):
    X = data.drop(columns=[target, 'date'], errors='ignore')
    y = data[target]
    cat_cols = X.select_dtypes(include=['object', 'string']).columns.tolist()
    for col in cat_cols:
        X[col] = X[col].astype('category')
    return X, y

X_a, y_a = get_X_y(section_data['a'])
print(f'Features shape: {X_a.shape}')
print('Feature columns:', list(X_a.columns))
print('Categorical features:', X_a.select_dtypes('category').columns.tolist())


Features shape: (371, 27)
Feature columns: ['meal_type', 'waste_organic_kg', 'waste_recyclable_kg', 'waste_landfill_kg', 'foot_traffic', 'is_holiday', 'has_special_event', 'year', 'month', 'day', 'day_of_week', 'day_of_year', 'week_of_year', 'quarter', 'is_weekend', 'dow_sin', 'dow_cos', 'doy_sin', 'doy_cos', 'lag_1', 'lag_7', 'lag_14', 'lag_28', 'rolling_mean_7', 'rolling_mean_14', 'rolling_std_7', 'rolling_max_7']
Categorical features: ['meal_type']


## Time-Based Train/Test Split

The last 30 days 90 rows = 3 meals × 30 days are reserved for testing.
The split is date-based so all meals of a given day remain in the same partition.


In [28]:
TEST_DAYS = 30

def make_splits(section_data: dict, test_days: int = TEST_DAYS) -> dict:
    """Return {section: {'train': df, 'test': df}} for all sections."""
    splits = {}
    for sec, data in section_data.items():
        data = data.sort_values('date').reset_index(drop=True)
        unique_dates = data['date'].drop_duplicates().sort_values()
        cutoff       = unique_dates.iloc[-test_days]
        train = data[data['date'] < cutoff].copy()
        test  = data[data['date'] >= cutoff].copy()
        splits[sec] = {'train': train, 'test': test}
        print(f'Section {sec.upper()} | Train: {len(train)} rows '
              f'({train["date"].nunique()} days) | '
              f'Test: {len(test)} rows ({test["date"].nunique()} days)')
    return splits


splits = make_splits(section_data)


Section A | Train: 292 rows (113 days) | Test: 79 rows (30 days)
Section B | Train: 294 rows (113 days) | Test: 78 rows (30 days)
Section C | Train: 297 rows (112 days) | Test: 75 rows (30 days)
Section D | Train: 288 rows (112 days) | Test: 74 rows (30 days)


## Helpers

In [29]:
def evaluate_model(model, X_test, y_test):
    """Return (RMSE, MAE, MAPE, R²) for a fitted model."""
    pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    mae  = mean_absolute_error(y_test, pred)
    mape = np.mean(np.abs((y_test - pred) /
                           np.where(y_test == 0, 1e-9, y_test))) * 100
    r2   = r2_score(y_test, pred)
    return rmse, mae, mape, r2


def evaluate_by_meal(model, test_data: pd.DataFrame) -> list:
    """Return per-meal metric dicts for a fitted model."""
    rows = []
    for meal in MEALS:
        mdf = test_data[test_data['meal_type'] == meal]
        if len(mdf) == 0:
            continue
        Xm, ym = get_X_y(mdf)
        rmse, mae, mape, r2 = evaluate_model(model, Xm, ym)
        rows.append({'meal': meal, 'RMSE': rmse, 'MAE': mae,
                     'MAPE': mape, 'R2': r2})
    return rows


## Baseline LightGBM Model (Default Parameters)

Train with defaults to establish a performance reference.


In [30]:
def train_lgb_baseline(X_train, y_train, cat_cols=None):
    """Fit LightGBM with default hyperparameters."""
    bl_model = lgb.LGBMRegressor(
        objective='regression',
        random_state=RANDOM_SEED,
        verbosity=-1,
        n_jobs=-1,
    )
    fit_params = {}
    if cat_cols:
        fit_params['categorical_feature'] = cat_cols
    bl_model.fit(X_train, y_train, **fit_params)
    return bl_model


baseline_results = []

for sec in SECTIONS:
    train = splits[sec]['train']
    test  = splits[sec]['test']
    X_train, y_train = get_X_y(train)
    X_test,  y_test  = get_X_y(test)
    cat_cols = X_train.select_dtypes('category').columns.tolist()

    bl_model = train_lgb_baseline(X_train, y_train, cat_cols)
    rmse, mae, mape, r2 = evaluate_model(bl_model, X_test, y_test)
    baseline_results.append({'section': sec.upper(), 'meal': 'overall',
                              'RMSE': rmse, 'MAE': mae,
                              'MAPE': mape, 'R2': r2})
    for row in evaluate_by_meal(bl_model, test):
        row['section'] = sec.upper()
        baseline_results.append(row)


baseline_df = pd.DataFrame(baseline_results)
print('Baseline Performance')
print(baseline_df.to_string(index=False))

Baseline Performance
section      meal     RMSE      MAE     MAPE       R2
      A   overall 0.302704 0.212030 6.161335 0.983482
      A breakfast 0.247247 0.156331 6.225095 0.967865
      A     lunch 0.422044 0.325918 8.483020 0.974963
      A    dinner 0.224294 0.175552 4.033968 0.991057
      B   overall 1.346583 0.433376 5.476226 0.869226
      B breakfast 0.101830 0.075096 3.437843 0.993715
      B     lunch 2.312452 1.017718 8.912012 0.753430
      B    dinner 0.277372 0.197869 4.033040 0.987399
      C   overall 0.590398 0.214263 3.530270 0.960396
      C breakfast 0.132674 0.090263 2.915560 0.993932
      C     lunch 1.011966 0.430054 4.024459 0.917571
      C    dinner 0.207998 0.134302 3.665164 0.990870
      D   overall 0.836059 0.351959 5.304417 0.924603
      D breakfast 0.300148 0.162362 4.252469 0.975319
      D     lunch 1.453976 0.794574 8.248137 0.834156
      D    dinner 0.180842 0.142720 3.711845 0.993122


## Cross-Validation and Hyperparameter Optimisation

OptunaBayesian search with TimeSeriesSplit (5 folds).
Up to 50 trials per section.

In [31]:
def lgb_objective(trial, X_train, y_train, cat_cols, cv_splits=5):
    """Optuna objective: minimise CV-RMSE for LightGBM."""
    params = {
        'num_leaves':        trial.suggest_int('num_leaves', 20, 150),
        'n_estimators':      trial.suggest_int('n_estimators', 100, 1000, step=50),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
        'subsample':         trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'objective':         'regression',
        'random_state':      RANDOM_SEED,
        'verbosity':         -1,
        'n_jobs':            -1,
    }

    tscv = TimeSeriesSplit(n_splits=cv_splits)
    rmse_scores = []

    for train_idx, val_idx in tscv.split(X_train):
        Xtr, Xval = X_train.iloc[train_idx], X_train.iloc[val_idx]
        ytr, yval = y_train.iloc[train_idx], y_train.iloc[val_idx]
        m = lgb.LGBMRegressor(**params)
        m.fit(Xtr, ytr, categorical_feature=cat_cols,
              callbacks=[lgb.early_stopping(50, verbose=False),
                         lgb.log_evaluation(-1)],
              eval_set=[(Xval, yval)])
        pred = m.predict(Xval)
        rmse_scores.append(np.sqrt(mean_squared_error(yval, pred)))

    return np.mean(rmse_scores)


N_TRIALS    = 50
best_models = {}
study_logs  = {}

for sec in SECTIONS:
    print(f"Optimising Section {sec.upper()}")
    train    = splits[sec]['train']
    X_train, y_train = get_X_y(train)
    cat_cols = X_train.select_dtypes('category').columns.tolist()

    study = optuna.create_study(
        direction='minimize',
        sampler=TPESampler(seed=RANDOM_SEED),
        pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=10),
    )

    def obj_wrapper(trial):
        return lgb_objective(trial, X_train, y_train, cat_cols)

    study.optimize(obj_wrapper, n_trials=N_TRIALS, show_progress_bar=True)
    study_logs[sec] = study.trials_dataframe()

    best_params = study.best_params
    print(f'Best params for Section {sec.upper()}: {best_params}')

    final_model = lgb.LGBMRegressor(
        **best_params,
        objective='regression',
        random_state=RANDOM_SEED,
        verbosity=-1,
        n_jobs=-1,
    )
    final_model.fit(X_train, y_train, categorical_feature=cat_cols)
    best_models[sec] = final_model

    with open(f'models/lightgbm_optimized/tuned_section_{sec}.pkl', 'wb') as f:
        pickle.dump(final_model, f)

    test = splits[sec]['test']
    X_test, y_test = get_X_y(test)
    rmse, mae, mape, r2 = evaluate_model(final_model, X_test, y_test)
    print(f'Test | RMSE: {rmse:.4f}  MAE: {mae:.4f}  '
          f'MAPE: {mape:.2f}%  R²: {r2:.4f}')

print('\nOptimization completed for all sections.')


Optimising Section A


Best trial: 47. Best value: 0.466889: 100%|██████████| 50/50 [01:02<00:00,  1.25s/it]


Best params for Section A: {'num_leaves': 129, 'n_estimators': 500, 'learning_rate': 0.035076459793632664, 'min_child_samples': 5, 'subsample': 0.7881065472372825, 'colsample_bytree': 0.8834063075700739, 'reg_lambda': 2.2587871909816033, 'reg_alpha': 0.24930482777551832}
Test | RMSE: 0.2337  MAE: 0.1268  MAPE: 3.26%  R²: 0.9902
Optimising Section B


Best trial: 47. Best value: 0.323135: 100%|██████████| 50/50 [00:58<00:00,  1.16s/it]


Best params for Section B: {'num_leaves': 129, 'n_estimators': 500, 'learning_rate': 0.035076459793632664, 'min_child_samples': 5, 'subsample': 0.7881065472372825, 'colsample_bytree': 0.8834063075700739, 'reg_lambda': 2.2587871909816033, 'reg_alpha': 0.24930482777551832}
Test | RMSE: 0.9470  MAE: 0.2682  MAPE: 3.63%  R²: 0.9353
Optimising Section C


Best trial: 44. Best value: 0.437289: 100%|██████████| 50/50 [01:09<00:00,  1.39s/it]


Best params for Section C: {'num_leaves': 141, 'n_estimators': 750, 'learning_rate': 0.06536388112674597, 'min_child_samples': 5, 'subsample': 0.6005800047171013, 'colsample_bytree': 0.7049193313408516, 'reg_lambda': 5.577701715879632, 'reg_alpha': 0.11553213211102557}
Test | RMSE: 0.2913  MAE: 0.1613  MAPE: 3.30%  R²: 0.9904
Optimising Section D


Best trial: 32. Best value: 0.390686: 100%|██████████| 50/50 [00:59<00:00,  1.20s/it]


Best params for Section D: {'num_leaves': 144, 'n_estimators': 450, 'learning_rate': 0.03361892784595996, 'min_child_samples': 5, 'subsample': 0.781373309090391, 'colsample_bytree': 0.7652469253764209, 'reg_lambda': 6.6114866717522345, 'reg_alpha': 0.3943203508652823}
Test | RMSE: 0.5233  MAE: 0.1830  MAPE: 2.91%  R²: 0.9705

Optimization completed for all sections.


## 10. Performance Comparison

In [32]:
final_results = []

for sec in SECTIONS:
    test = splits[sec]['test']
    X_test, y_test = get_X_y(test)

    b_rmse, b_mae, b_mape, b_r2 = evaluate_model(bl_model, X_test, y_test)

    tuned_model = best_models[sec]
    t_rmse, t_mae, t_mape, t_r2 = evaluate_model(tuned_model, X_test, y_test)

    for model_name, rmse, mae, mape, r2 in [
        ('Baseline',       b_rmse, b_mae, b_mape, b_r2),
        ('Tuned (Optuna)', t_rmse, t_mae, t_mape, t_r2),
    ]:
        final_results.append({'section': sec.upper(), 'model': model_name,
                               'meal': 'overall', 'RMSE': rmse,
                               'MAE': mae, 'MAPE': mape, 'R2': r2})

    for row in evaluate_by_meal(tuned_model, test):
        row.update({'section': sec.upper(), 'model': 'Tuned (Optuna)'})
        final_results.append(row)

results_df = pd.DataFrame(final_results)
print('Performance Comparison')
print(results_df.to_string(index=False))

results_df.to_csv('metrics/lightgbm_model_comparison.csv', index=False)

Performance Comparison
section          model      meal     RMSE      MAE     MAPE       R2
      A       Baseline   overall 0.254043 0.160524 4.490785 0.988366
      A Tuned (Optuna)   overall 0.233663 0.126820 3.263136 0.990158
      A Tuned (Optuna) breakfast 0.136645 0.080678 2.967357 0.990185
      A Tuned (Optuna)     lunch 0.351891 0.195869 4.316037 0.982595
      A Tuned (Optuna)    dinner 0.186579 0.118980 2.673007 0.993812
      B       Baseline   overall 1.507348 0.427207 5.078602 0.836136
      B Tuned (Optuna)   overall 0.947038 0.268151 3.625004 0.935317
      B Tuned (Optuna) breakfast 0.060847 0.047893 2.394726 0.997756
      B Tuned (Optuna)     lunch 1.610318 0.557011 4.932294 0.880431
      B Tuned (Optuna)    dinner 0.295599 0.188717 3.465615 0.985688
      C       Baseline   overall 0.962015 0.339034 4.767425 0.894849
      C Tuned (Optuna)   overall 0.291266 0.161302 3.300625 0.990361
      C Tuned (Optuna) breakfast 0.105578 0.080641 2.980469 0.996157
      C Tun

## 7-Day Forecast (Breakfast, Lunch, Dinner)

In [33]:
def update_temporal_features(row: pd.Series, next_date: pd.Timestamp) -> pd.Series:
    row = row.copy()
    row['date']         = next_date
    row['year']         = next_date.year
    row['month']        = next_date.month
    row['day']          = next_date.day
    row['day_of_week']  = next_date.dayofweek
    row['day_of_year']  = next_date.dayofyear
    row['week_of_year'] = int(next_date.isocalendar().week)
    row['quarter']      = next_date.quarter
    row['is_weekend']   = int(next_date.dayofweek >= 5)
    row['dow_sin']      = np.sin(2 * np.pi * next_date.dayofweek / 7)
    row['dow_cos']      = np.cos(2 * np.pi * next_date.dayofweek / 7)
    row['doy_sin']      = np.sin(2 * np.pi * next_date.dayofyear / 365)
    row['doy_cos']      = np.cos(2 * np.pi * next_date.dayofyear / 365)
    return row


def recompute_lags(temp_df: pd.DataFrame, target: str = TARGET) -> pd.DataFrame:
    for lag in [1, 7, 14, 28]:
        temp_df[f'lag_{lag}'] = temp_df[target].shift(lag)
    temp_df['rolling_mean_7']  = temp_df[target].shift(1).rolling(7,  min_periods=1).mean()
    temp_df['rolling_mean_14'] = temp_df[target].shift(1).rolling(14, min_periods=1).mean()
    temp_df['rolling_std_7']   = temp_df[target].shift(1).rolling(7,  min_periods=1).std().fillna(0)
    temp_df['rolling_max_7']   = temp_df[target].shift(1).rolling(7,  min_periods=1).max()
    return temp_df


def forecast_next_week(model, history_df: pd.DataFrame,
                       horizon: int = 7, target: str = TARGET) -> dict:
    MAX_LOOKBACK = 28
    forecasts    = {meal: [] for meal in MEALS}

    history_df = history_df.sort_values(['date', 'meal_type']).reset_index(drop=True)
    last_date  = history_df['date'].max()

    meal_histories = {
        meal: history_df[history_df['meal_type'] == meal].copy()
        for meal in MEALS
    }

    for i in range(1, horizon + 1):
        next_date = last_date + pd.Timedelta(days=i)

        for meal in MEALS:
            meal_hist = meal_histories[meal]

            new_row           = meal_hist.iloc[-1:].copy()
            new_row           = new_row.apply(
                lambda r: update_temporal_features(r, next_date), axis=1)
            new_row[target]   = np.nan
            new_row['meal_type'] = meal

            temp = pd.concat([meal_hist.iloc[-MAX_LOOKBACK:], new_row],
                             ignore_index=True)
            temp = recompute_lags(temp, target)

            X_pred = temp.iloc[-1:].drop(columns=[target, 'date'], errors='ignore')
            for col in X_pred.select_dtypes(include=['object', 'string']).columns:
                X_pred[col] = X_pred[col].astype('category')
            predicted = float(model.predict(X_pred)[0])
            forecasts[meal].append(predicted)

            new_row[target] = predicted
            meal_histories[meal] = pd.concat([meal_hist, new_row],
                                             ignore_index=True)
    return forecasts


In [34]:
weekly_forecasts = {}

for sec in SECTIONS:
    with open(f'models/lightgbm_optimized/tuned_section_{sec}.pkl', 'rb') as f:
        model = pickle.load(f)

    train_history = splits[sec]['train']
    forecasts     = forecast_next_week(model, train_history, horizon=7)
    weekly_forecasts[sec.upper()] = forecasts

    print(f'\nSection {sec.upper()} 7-day forecast:')
    for meal, preds in forecasts.items():
        print(f'  {meal:10s}: {[round(p, 2) for p in preds]}')

forecast_rows = [
    {'section': sec, 'meal_type': meal, 'day': f'Day_{d}', 'forecast_kg': val}
    for sec, meal_dict in weekly_forecasts.items()
    for meal, preds in meal_dict.items()
    for d, val in enumerate(preds, start=1)
]
forecast_df = pd.DataFrame(forecast_rows)
forecast_df.to_csv('metrics/lightgbm_weekly_forecasts.csv', index=False)
print('\nWeekly forecasts saved to lightgbm_weekly_forecasts.csv')



Section A 7-day forecast:
  breakfast : [1.31, 1.31, 1.3, 1.32, 1.31, 1.31, 1.32]
  lunch     : [5.63, 5.58, 5.61, 5.63, 5.63, 5.67, 5.63]
  dinner    : [1.64, 1.64, 1.66, 1.64, 1.65, 1.65, 1.64]

Section B 7-day forecast:
  breakfast : [2.57, 2.57, 2.56, 2.57, 2.58, 2.58, 2.57]
  lunch     : [5.94, 5.94, 5.91, 5.95, 5.91, 5.92, 5.95]
  dinner    : [7.9, 7.87, 7.88, 7.93, 7.95, 7.89, 7.95]

Section C 7-day forecast:
  breakfast : [4.22, 4.1, 4.09, 4.1, 4.09, 4.1, 4.09]
  lunch     : [13.59, 13.15, 13.35, 13.25, 13.11, 13.08, 13.08]
  dinner    : [10.51, 10.14, 10.18, 10.14, 10.15, 10.17, 10.17]

Section D 7-day forecast:
  breakfast : [1.04, 1.07, 1.05, 1.06, 1.05, 1.06, 1.07]
  lunch     : [2.07, 2.09, 2.08, 2.07, 2.06, 2.07, 2.07]
  dinner    : [2.42, 2.42, 2.41, 2.42, 2.43, 2.4, 2.41]

Weekly forecasts saved to lightgbm_weekly_forecasts.csv


## 12. Reload Saved Models and Verify Predictions

In [36]:
for sec in SECTIONS:
    with open(f'models/lightgbm_optimized/tuned_section_{sec}.pkl', 'rb') as f:
        reloaded = pickle.load(f)

    test = splits[sec]['test']
    X_test, y_test = get_X_y(test)
    in_mem  = evaluate_model(best_models[sec],  X_test, y_test)[0]
    reloaded_rmse = evaluate_model(reloaded, X_test, y_test)[0]
    match = 'Match' if abs(in_mem - reloaded_rmse) < 1e-9 else 'Mismatch'
    print(f'Section {sec.upper()} | RMSE: {in_mem:.6f} | '
          f'Reloaded RMSE: {reloaded_rmse:.6f} | {match}')


Section A | RMSE: 0.233663 | Reloaded RMSE: 0.233663 | Match
Section B | RMSE: 0.947038 | Reloaded RMSE: 0.947038 | Match
Section C | RMSE: 0.291266 | Reloaded RMSE: 0.291266 | Match
Section D | RMSE: 0.523347 | Reloaded RMSE: 0.523347 | Match


## Forecasts: Baseline vs Tuned, by Meal Type

In [38]:
all_comparison = {}

for sec in SECTIONS:
    train_data = splits[sec]['train']
    base_preds = forecast_next_week(bl_model, train_data.copy(), horizon=7)
    with open(f'models/lightgbm_optimized/tuned_section_{sec}.pkl', 'rb') as f:
        tuned_model = pickle.load(f)
    tuned_preds = forecast_next_week(tuned_model, train_data.copy(), horizon=7)
    all_comparison[sec.upper()] = {'Baseline': base_preds, 'Tuned': tuned_preds}

print('Forecasts compiled for all sections.')

Forecasts compiled for all sections.


In [39]:
last_train_date = splits['a']['train']['date'].max()
forecast_dates  = pd.date_range(start=last_train_date + pd.Timedelta(days=1), periods=7)

plot_rows = []
for sec, model_dict in all_comparison.items():
    for model_name, meal_dict in model_dict.items():
        for meal, preds in meal_dict.items():
            for date, val in zip(forecast_dates, preds):
                plot_rows.append({
                    'Date': date,
                    'Section': sec,
                    'Model': model_name,
                    'Meal': meal,
                    'Forecast': val
                })

forecast_df_plot = pd.DataFrame(plot_rows)
display(forecast_df_plot.head(12))

,Date,Section,Model,Meal,Forecast
0,2025-06-01,A,Baseline,breakfast,1.476393
1,2025-06-02,A,Baseline,breakfast,1.478321
2,2025-06-03,A,Baseline,breakfast,1.478061
3,2025-06-04,A,Baseline,breakfast,1.476028
4,2025-06-05,A,Baseline,breakfast,1.494355
5,2025-06-06,A,Baseline,breakfast,1.501375
6,2025-06-07,A,Baseline,breakfast,1.481682
7,2025-06-01,A,Baseline,lunch,5.633241
8,2025-06-02,A,Baseline,lunch,5.625908
9,2025-06-03,A,Baseline,lunch,5.618487
